# MiniProyecto 2 - Clasificacion de niveles de obesidad


Este notebook resume el flujo del proyecto: carga del dataset, depuracion, modelado y comparacion de resultados.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, matthews_corrcoef


In [ ]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = ROOT / 'data' / 'raw' / 'obesity.csv'
TARGET_COLUMN = 'NObeyesdad'
df = pd.read_csv(DATA_PATH)
df.head()


## Resumen y depuracion


In [ ]:
print(df.shape)
print(df[TARGET_COLUMN].value_counts())
print('Valores faltantes totales:', int(df.isna().sum().sum()))
print('Duplicados exactos:', int(df.duplicated().sum()))
df = df.drop_duplicates().reset_index(drop=True)
print('Forma tras eliminar duplicados:', df.shape)


## Preparacion de variables


In [ ]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]
numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(exclude='number').columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape


## Entrenamiento de modelos


In [ ]:
models = {
    'logistic_regression': (
        LogisticRegression(max_iter=4000, random_state=42),
        {'model__C':[0.1, 1, 3, 10], 'model__solver':['lbfgs', 'newton-cg']},
    ),
    'svm_rbf': (
        SVC(random_state=42),
        {'model__C':[0.5, 1, 3, 10], 'model__gamma':['scale', 0.1, 0.03, 0.01], 'model__kernel':['rbf']},
    ),
    'decision_tree': (
        DecisionTreeClassifier(random_state=42),
        {'model__criterion':['gini', 'entropy'], 'model__max_depth':[None, 5, 10, 20], 'model__min_samples_split':[2, 5, 10], 'model__min_samples_leaf':[1, 2, 4]},
    ),
    'random_forest': (
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {'model__n_estimators':[200, 400], 'model__max_depth':[None, 10, 20], 'model__min_samples_split':[2, 5], 'model__min_samples_leaf':[1, 2]},
    ),
}
rows = []
for name, (model, params) in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    search = GridSearchCV(pipeline, params, cv=5, scoring='f1_macro', n_jobs=-1)
    search.fit(X_train, y_train)
    preds = search.best_estimator_.predict(X_test)
    rows.append({
        'model': name,
        'cv_f1_macro': search.best_score_,
        'test_accuracy': accuracy_score(y_test, preds),
        'test_balanced_accuracy': balanced_accuracy_score(y_test, preds),
        'test_f1_macro': f1_score(y_test, preds, average='macro'),
        'test_mcc': matthews_corrcoef(y_test, preds),
    })
results_df = pd.DataFrame(rows).sort_values(['test_mcc', 'test_f1_macro'], ascending=False)
results_df


## Resultados exportados

El script `src/run_analysis.py` genera automaticamente el informe final, las figuras, los CSV de metricas y las matrices de confusion.
